# ES 模块

学习目标：拆分和组合模块接口，理解实时绑定与模块求值，按宿主条件导入本地 JavaScript 和 JSON。

前置知识：函数、对象属性、词法作用域、暂时性死区、严格模式与 JSON。

适用版本：ECMAScript 2025；Node.js 24.11.0 原生 .mjs；导入属性与 JSON 模块使用该版本稳定支持的 with 语法。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/19-es-modules/。

1. [math.mjs](scripts/19-es-modules/math.mjs)：具名与默认导出的提供方。
2. [basic.mjs](scripts/19-es-modules/basic.mjs)：默认导入与具名别名。
3. [missing-export-error.mjs](scripts/19-es-modules/missing-export-error.mjs)：导入不存在的具名接口。
4. [module-scope.mjs](scripts/19-es-modules/module-scope.mjs)：顶层声明和 this。
5. [strict-error.mjs](scripts/19-es-modules/strict-error.mjs)：模块不允许意外创建全局变量。
6. [counter.mjs](scripts/19-es-modules/counter.mjs)：可变导出和默认表达式。
7. [live-bindings.mjs](scripts/19-es-modules/live-bindings.mjs)：读取实时值与修改导出对象。
8. [import-assignment-error.mjs](scripts/19-es-modules/import-assignment-error.mjs)：导入方不能重绑定。
9. [namespace.mjs](scripts/19-es-modules/namespace.mjs)：命名空间的结构和实时读取。
10. [namespace-write-error.mjs](scripts/19-es-modules/namespace-write-error.mjs)：命名空间属性拒绝写入。
11. [public-api.mjs](scripts/19-es-modules/public-api.mjs)：集中转发模块接口。
12. [reexport.mjs](scripts/19-es-modules/reexport.mjs)：使用公共入口。
13. [registration.mjs](scripts/19-es-modules/registration.mjs)：可观察的顶层副作用。
14. [first-client.mjs](scripts/19-es-modules/first-client.mjs)：第一条依赖路径。
15. [second-client.mjs](scripts/19-es-modules/second-client.mjs)：第二条依赖路径。
16. [side-effects.mjs](scripts/19-es-modules/side-effects.mjs)：两条路径共享一次求值。
17. [cycle-a.mjs](scripts/19-es-modules/cycle-a.mjs)：循环提供方 A，延后读取。
18. [cycle-b.mjs](scripts/19-es-modules/cycle-b.mjs)：循环提供方 B，延后读取。
19. [cycle-safe.mjs](scripts/19-es-modules/cycle-safe.mjs)：双方初始化后调用。
20. [cycle-bad-b.mjs](scripts/19-es-modules/cycle-bad-b.mjs)：在循环顶层提前读取。
21. [cycle-error.mjs](scripts/19-es-modules/cycle-error.mjs)：循环初始化反例入口。
22. [module-location.mjs](scripts/19-es-modules/module-location.mjs)：相对当前模块定位资源。
23. [extension-error.mjs](scripts/19-es-modules/extension-error.mjs)：Node ESM 不自动补扩展名。
24. [settings.json](scripts/19-es-modules/settings.json)：被导入的 JSON 数据。
25. [json-module.mjs](scripts/19-es-modules/json-module.mjs)：带类型属性导入 JSON。
26. [json-attribute-error.mjs](scripts/19-es-modules/json-attribute-error.mjs)：JSON 导入缺少类型属性。
27. [json-named-error.mjs](scripts/19-es-modules/json-named-error.mjs)：JSON 不提供同名字段导出。

## 1 用导出声明模块接口

一个模块只通过明确导出的名称提供接口。具名导出可有多个，导入时名称必须对应；as 在导出端或导入端建立别名。每个模块至多有一个 default 导出，默认导入的本地名称由使用者决定，花括号不是对象解构语法。

静态 import/export 只能位于模块顶层，模块说明符是字符串字面量，不能放在 if 或普通函数中按条件执行。本例 .mjs 由 Node 明确识别为 ES 模块，math.mjs 提供函数，basic.mjs 是运行入口。路径相对导入文件解析，而非相对终端工作目录。

[math.mjs](scripts/19-es-modules/math.mjs)：

```javascript
export const minutesPerHour = 60;
function add(left, right) {
  return left + right;
}
export { add as sum };
export default function describe(minutes) {
  return String(minutes) + " 分钟";
}
```

[basic.mjs](scripts/19-es-modules/basic.mjs)：

```javascript
import label, { sum as total, minutesPerHour } from "./math.mjs";
console.log(label(total(minutesPerHour, 30)));

// 按本例输入运行，输出依次为：
// 90 分钟
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/basic.mjs
```

[missing-export-error.mjs](scripts/19-es-modules/missing-export-error.mjs)：

```javascript
import { add } from "./math.mjs";
console.log(add(1, 2));

// 独立运行：退出状态为 1；诊断包含 SyntaxError；does not provide an export named 'add'。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/missing-export-error.mjs
```

## 2 模块作用域与严格模式

模块顶层声明属于模块作用域，即使用 var 也不会成为 globalThis 的属性。模块代码自动采用严格模式，顶层 this 为 undefined；普通函数裸调用也保持严格模式的 this 规则。

严格模式下向未声明变量赋值会抛 ReferenceError。这里的模块作用域属于语言机制，globalThis 中有哪些 API 则取决于宿主；把代码从浏览器移到 Node 并不会自动获得 DOM。

[module-scope.mjs](scripts/19-es-modules/module-scope.mjs)：

```javascript
var moduleOnlyValue = 7;
function getThis() { return this; }
console.log(moduleOnlyValue, Object.hasOwn(globalThis, "moduleOnlyValue"));
console.log(this === undefined, getThis() === undefined);

// 按本例输入运行，输出依次为：
// 7 false
// true true
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/module-scope.mjs
```

[strict-error.mjs](scripts/19-es-modules/strict-error.mjs)：

```javascript
undeclaredLessonTotal = 3;

// 独立运行：退出状态为 1；诊断包含 ReferenceError；undeclaredLessonTotal is not defined。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/strict-error.mjs
```

## 3 实时绑定与对象可变性

导入建立实时绑定（live binding），读取时取得导出方绑定的当前值；它不是每次导入时复制一次的普通局部变量。导入方不能重新赋值该绑定，但导出的对象仍可能允许修改属性，绑定只读不等于对象不可变。

默认导出也有绑定。需要特别区分两种导出声明：export { count as default } 把默认名称连接到已有 count；export default count 则在执行该表达式时取值并存入独立的默认绑定。下例同时展示原始数值的表达式默认导出和实时具名导出，snapshot 不随 count 后续变化。

[counter.mjs](scripts/19-es-modules/counter.mjs)：

```javascript
export let count = 0;
export const settings = { title: "JS" };
export function increment() { count += 1; }
export default count;
```

[live-bindings.mjs](scripts/19-es-modules/live-bindings.mjs)：

```javascript
import snapshot, { count, increment, settings } from "./counter.mjs";
console.log(snapshot, count);
increment();
settings.title = "模块";
console.log(snapshot, count, settings.title);

// 按本例输入运行，输出依次为：
// 0 0
// 0 1 模块
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/live-bindings.mjs
```

[import-assignment-error.mjs](scripts/19-es-modules/import-assignment-error.mjs)：

```javascript
import { count } from "./counter.mjs";
count = 10;

// 独立运行：退出状态为 1；诊断包含 TypeError；Assignment to constant variable。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/import-assignment-error.mjs
```

## 4 命名空间对象与再导出

import * as namespace 取得模块命名空间对象；namespace 是本例本地名称，它的属性对应模块导出，包括 default。该对象的原型为 null、不可扩展，读取属性仍观察实时绑定。它不是导出值的普通快照，也不能由导入方给属性赋值；即使属性描述符显示 writable: true，其特殊内部写入操作仍拒绝赋值。

再导出可以组织公共入口。export * from 转发具名导出但不转发 default；默认值需明确转发。export * as math 则转发整个命名空间。再导出不会顺带在当前模块创建同名本地变量；需要本地调用时另写 import。多个星号导出发生同名歧义时，应使用明确的具名转发消除歧义。

[namespace.mjs](scripts/19-es-modules/namespace.mjs)：

```javascript
import * as counter from "./counter.mjs";
console.log(Object.keys(counter).join(","));
console.log(Object.getPrototypeOf(counter) === null, Object.isExtensible(counter));
console.log(Object.getOwnPropertyDescriptor(counter, "count").writable);
counter.increment();
console.log(counter.count, counter.default);

// 按本例输入运行，输出依次为：
// count,default,increment,settings
// true false
// true
// 1 0
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/namespace.mjs
```

[namespace-write-error.mjs](scripts/19-es-modules/namespace-write-error.mjs)：

```javascript
import * as counter from "./counter.mjs";
counter.count = 10;

// 独立运行：退出状态为 1；诊断包含 TypeError；Cannot assign to read only property 'count'。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/namespace-write-error.mjs
```

[public-api.mjs](scripts/19-es-modules/public-api.mjs)：

```javascript
export * from "./math.mjs";
export { default as describe } from "./math.mjs";
export * as math from "./math.mjs";
```

[reexport.mjs](scripts/19-es-modules/reexport.mjs)：

```javascript
import * as api from "./public-api.mjs";
console.log(Object.keys(api).join(","));
console.log(api.describe(api.sum(5, 10)), api.math.minutesPerHour);
console.log(Object.hasOwn(api, "default"));

// 按本例输入运行，输出依次为：
// describe,math,minutesPerHour,sum
// 15 分钟 60
// false
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/reexport.mjs
```

## 5 依赖求值与导入副作用

静态导入先建立依赖关系，再按模块图加载、连接并求值；不能用 import 在源码里的行号推断为普通函数调用顺序。在本节没有循环和顶层 await 的图中，依赖模块先于入口模块主体求值。

只写 import 加说明符会运行依赖的顶层代码，即“副作用导入”。同一个模块实例在同一加载环境中成功求值一次后不会因重复导入重跑；Node 按解析后的 URL 识别 ES 模块，不同查询参数或片段可能形成不同实例，不能把“一次”理解为整台电脑永久只执行一次。

下例两个中间模块依赖同一注册模块，只打印一次注册。真实应用宜把创建资源等操作暴露为显式函数，让调用方决定生命周期，减少导入时隐藏的动作。

[registration.mjs](scripts/19-es-modules/registration.mjs)：

```javascript
console.log("注册模块"); // → 注册模块；同一模块实例被两处导入时只打印一次。
```

[first-client.mjs](scripts/19-es-modules/first-client.mjs)：

```javascript
import "./registration.mjs";
export const first = "甲";
```

[second-client.mjs](scripts/19-es-modules/second-client.mjs)：

```javascript
import "./registration.mjs";
export const second = "乙";
```

[side-effects.mjs](scripts/19-es-modules/side-effects.mjs)：

```javascript
console.log("入口主体");
import { first } from "./first-client.mjs";
import { second } from "./second-client.mjs";
console.log(first + second);

// 按本例输入运行，输出依次为：
// 注册模块
// 入口主体
// 甲乙
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/side-effects.mjs
```

## 6 循环依赖与初始化时机

A 导入 B、B 再导入 A 就形成循环。循环本身不必然报错：关键是读取导出绑定时，它是否已经初始化。连接阶段可建立绑定关系，let/const 的值仍要到求值执行其声明时才初始化；提前读取会触发暂时性死区。

第一个图通过函数把读取推迟到双方初始化后才进行，因此能完成；第二个图在 B 顶层立刻读取尚未初始化的 A，得到 ReferenceError。不要靠交换 import 行顺序维持这种脆弱关系。更清楚的结构通常是把公共数据移到第三个模块，或合并职责紧密的模块。

[cycle-a.mjs](scripts/19-es-modules/cycle-a.mjs)：

```javascript
import { readB } from "./cycle-b.mjs";
export const a = "A";
export function combined() { return a + readB(); }
```

[cycle-b.mjs](scripts/19-es-modules/cycle-b.mjs)：

```javascript
import { a } from "./cycle-a.mjs";
export const b = "B";
export function readB() { return b + a; }
```

[cycle-safe.mjs](scripts/19-es-modules/cycle-safe.mjs)：

```javascript
import { combined } from "./cycle-a.mjs";
console.log(combined());

// 按本例输入运行，输出依次为：
// ABA
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/cycle-safe.mjs
```

[cycle-bad-b.mjs](scripts/19-es-modules/cycle-bad-b.mjs)：

```javascript
import { a } from "./cycle-error.mjs";
export const b = a + "B";
```

[cycle-error.mjs](scripts/19-es-modules/cycle-error.mjs)：

```javascript
import { b } from "./cycle-bad-b.mjs";
export const a = "A";
console.log(b);

// 独立运行：退出状态为 1；诊断包含 ReferenceError；Cannot access 'a' before initialization。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/cycle-error.mjs
```

## 7 import.meta 与宿主解析条件

import.meta 是仅在模块中可用的元信息入口；语言规定它的创建机制，具体属性由宿主提供。Node 的 import.meta.url 是当前模块的完整 URL。结合 new URL 和完整相对后缀，可相对模块定位资源，避免受终端工作目录影响；URL 是宿主接口，不把 Windows 磁盘路径直接当 URL 拼接。

Node.js 24.11.0 原生 ESM 的相对或绝对文件导入要写完整扩展名，也不自动补目录的 index 文件；裸说明符按包解析规则和包入口配置处理，node: 前缀用于 Node 内置模块。不能把 CommonJS 的补扩展名习惯套过来。

浏览器由 HTML 的 module 脚本入口加载模块，以 URL 解析相对说明符，默认不按 Node 的 node_modules 规则查找包；裸说明符通常需要 import map 映射。响应须满足模块类型的 MIME 与跨源加载条件，本地预览使用技术 README 的 HTTP 入口。浏览器 JSON 模块还须核查浏览器版本及响应类型；Node 成功运行并不证明浏览器具备相同支持。本章源文件入口在 Node 执行，浏览器解析条件用于说明宿主边界。

[module-location.mjs](scripts/19-es-modules/module-location.mjs)：

```javascript
const resource = new URL("./settings.json", import.meta.url);
console.log(new URL(import.meta.url).protocol);
console.log(resource.pathname.endsWith("/scripts/19-es-modules/settings.json"));
console.log(resource.protocol === "file:");

// 按本例输入运行，输出依次为：
// file:
// true
// true
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/module-location.mjs
```

[extension-error.mjs](scripts/19-es-modules/extension-error.mjs)：

```javascript
import { sum } from "./math";
console.log(sum(1, 2));

// 独立运行：退出状态为 1；诊断包含 ERR_MODULE_NOT_FOUND；Cannot find module。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/extension-error.mjs
```

## 8 导入属性与 JSON 模块

导入属性（import attributes）在模块请求上声明附加条件；with { type: "json" } 表示按 JSON 模块处理。属性值须为字符串字面量，属性名不能重复，宿主还会检查支持的属性。它与普通 JavaScript 对象参数不是同一种语法。

Node.js 24.11.0 稳定支持 JSON 模块，type: "json" 是必需属性；只写 .json 扩展名仍会失败。JSON 模块只提供 default 导出，其值是解析后的数据，不会为每个 JSON 键建立具名导出。数据对象也不会自动深冻结，业务校验规则仍适用。

本章采用 with，不使用旧式 assert 导入断言。动态 import() 是返回 Promise 的表达式，适合按需加载；其错误与异步流程将在掌握 Promise、async/await 后展开，本章只建立静态模块图。

[settings.json](scripts/19-es-modules/settings.json)：

```json
{
  "title": "模块",
  "chapters": 3
}
```

[json-module.mjs](scripts/19-es-modules/json-module.mjs)：

```javascript
import settings from "./settings.json" with { type: "json" };
console.log(settings.title, settings.chapters);

// 按本例输入运行，输出依次为：
// 模块 3
```

Step 1：运行本节示例。

```bash
node scripts/19-es-modules/json-module.mjs
```

[json-attribute-error.mjs](scripts/19-es-modules/json-attribute-error.mjs)：

```javascript
import settings from "./settings.json";
console.log(settings.title);

// 独立运行：退出状态为 1；诊断包含 ERR_IMPORT_ATTRIBUTE_MISSING；needs an import attribute of "type: json"。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/json-attribute-error.mjs
```

[json-named-error.mjs](scripts/19-es-modules/json-named-error.mjs)：

```javascript
import { title } from "./settings.json" with { type: "json" };
console.log(title);

// 独立运行：退出状态为 1；诊断包含 SyntaxError；does not provide an export named 'title'。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/19-es-modules/json-named-error.mjs
```

## 本章小结

- 模块接口由导出声明决定，默认导出与具名导入有不同语法。
- 导入是不可重绑定的实时连接，模块命名空间也不接受普通属性写入。
- 循环依赖要分析初始化时机，副作用要分析模块实例与依赖图。
- 文件解析、import.meta 属性和 JSON 加载条件须按宿主与版本核查。

## 练习

1. 将数学模块的默认函数转发为 formatDuration，并保留 sum。可核对标准：消费方只依赖公共入口，原模块没有被复制，formatDuration(90) 返回 90 分钟。
2. 给 counter 增加 reset 函数。可核对标准：调用后所有实时导入读到 0，默认表达式的快照含义不变；消费方不直接给 count 赋值。
3. 把失败的循环示例改为无环依赖。可核对标准：公共值放在第三个模块，入口能打印 AB，模块顶层没有提前读取未初始化绑定。
4. 增加 JSON 字段并导入。可核对标准：保留 type 属性，通过默认导入对象读取字段；说明浏览器还需核查的两项宿主条件。

## 参考与引用来源

- TC39 官方 ECMAScript 2025 分页版：[§16.2 模块图、求值、Imports、Exports 和 JSON 模块](https://tc39.es/ecma262/2025/multipage/ecmascript-language-scripts-and-modules.html#sec-modules)；[§9.1.1.5 模块绑定与 this](https://tc39.es/ecma262/2025/multipage/executable-code-and-execution-contexts.html#sec-module-environment-records)；[§10.4.6 模块命名空间对象](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-module-namespace-exotic-objects)。
- Node.js：[24.11.0：Mandatory file extensions、URLs、import.meta、Import attributes 与 JSON modules](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html)。
- WHATWG：[浏览器模块说明符解析与 import map](https://html.spec.whatwg.org/multipage/webappapis.html#resolve-a-module-specifier)；[模块抓取、类型与跨源条件](https://html.spec.whatwg.org/multipage/webappapis.html#fetch-a-single-module-script)。
- MDN 用法对照：[默认导出、再导出、副作用与 Cyclic imports](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Modules)。